# Configurable Runnables

The `configurable.py` module defines serializable Runnable wrappers whose fields or underlying implementations can be selected dynamically through `RunnableConfig`.

A configurable Runnable can expose individual fields as runtime options or allow one Runnable to be selected from several alternatives. The selected Runnable is prepared before invocation, batching, streaming, transformation, schema generation, or graph generation.

# DynamicRunnable

`DynamicRunnable` is the abstract base class for serializable Runnables that can be configured dynamically.

It wraps a default `RunnableSerializable` and resolves the actual Runnable to execute from the supplied configuration.

## Bases

- `RunnableSerializable[Input, Output]`

## Attributes

1. `default`: Stores the default Runnable used when no alternative configuration is supplied.
   * **Type:**
     ```python
     default: RunnableSerializable[Input, Output]
     ```

2. `config`: Stores configuration bound to the dynamic Runnable.
   * **Type:**
     ```python
     config: RunnableConfig | None = None
     ```

## Configuration

1. `model_config`: Allows arbitrary Python types in the Pydantic model.
   * **Definition:**
     ```python
     model_config = ConfigDict(
         arbitrary_types_allowed=True
     )
     ```

### Properties

1. `InputType`: Returns the input type accepted by the default Runnable.
   * **Type:**
     ```python
     InputType: type[Input]
     ```

2. `OutputType`: Returns the output type produced by the default Runnable.
   * **Type:**
     ```python
     OutputType: type[Output]
     ```

### Methods

1. `is_lc_serializable`: Indicates that the dynamic Runnable supports LangChain serialization.
   * **Syntax:**
     ```python
     @classmethod
     is_lc_serializable(
         cls
     ) -> bool
     ```

2. `get_lc_namespace`: Returns the LangChain serialization namespace.
   * **Syntax:**
     ```python
     @classmethod
     get_lc_namespace(
         cls
     ) -> list[str]
     ```

3. `get_input_schema`: Returns the input schema of the Runnable selected by the supplied configuration.
   * **Syntax:**
     ```python
     get_input_schema(
         self,
         config: RunnableConfig | None = None # Configuration used to select the Runnable
     ) -> TypeBaseModel
     ```

4. `get_output_schema`: Returns the output schema of the Runnable selected by the supplied configuration.
   * **Syntax:**
     ```python
     get_output_schema(
         self,
         config: RunnableConfig | None = None # Configuration used to select the Runnable
     ) -> TypeBaseModel
     ```

5. `get_graph`: Returns the graph representation of the Runnable selected by the supplied configuration.
   * **Syntax:**
     ```python
     get_graph(
         self,
         config: RunnableConfig | None = None # Configuration used to select the Runnable
     ) -> Graph
     ```

6. `with_config`: Binds configuration to the dynamic Runnable and returns a new Runnable wrapper.

   The supplied configuration and additional keyword values are merged before being stored.

   * **Syntax:**
     ```python
     with_config(
         self,
         config: RunnableConfig | None = None, # Configuration to bind
         **kwargs: Any # Additional configuration values
     ) -> Runnable[Input, Output]
     ```

7. `prepare`: Resolves nested dynamic Runnables into the concrete Runnable that should be executed.

   Bound configuration and invocation configuration are merged at every dynamic layer.

   * **Syntax:**
     ```python
     prepare(
         self,
         config: RunnableConfig | None = None # Configuration used to prepare the Runnable
     ) -> tuple[
         Runnable[Input, Output],
         RunnableConfig
     ]
     ```

8. `_prepare`: Defines how a dynamic Runnable selects or constructs its concrete Runnable.

   Subclasses must implement this protected abstract method.

   * **Syntax:**
     ```python
     @abstractmethod
     _prepare(
         self,
         config: RunnableConfig | None = None # Configuration used to prepare the Runnable
     ) -> tuple[
         Runnable[Input, Output],
         RunnableConfig
     ]
     ```

9. `invoke`: Resolves the configured Runnable and synchronously transforms one input into an output.
   * **Syntax:**
     ```python
     invoke(
         self,
         input: Input, # Input passed to the selected Runnable
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional invocation arguments
     ) -> Output
     ```

10. `ainvoke`: Resolves the configured Runnable and asynchronously transforms one input into an output.
    * **Syntax:**
      ```python
      async ainvoke(
          self,
          input: Input, # Input passed to the selected Runnable
          config: RunnableConfig | None = None, # Runtime configuration
          **kwargs: Any # Additional invocation arguments
      ) -> Output
      ```

11. `batch`: Resolves a Runnable for every input and processes the inputs synchronously.

    When every configuration resolves to the default Runnable, execution is delegated directly to its optimized `batch` implementation. Otherwise, prepared Runnables are invoked using the configured executor.

    * **Syntax:**
      ```python
      batch(
          self,
          inputs: list[Input], # Inputs to process
          config: RunnableConfig
          | list[RunnableConfig]
          | None = None, # Configuration for one or more inputs
          *,
          return_exceptions: bool = False, # Return exceptions instead of raising them
          **kwargs: Any | None # Additional invocation arguments
      ) -> list[Output]
      ```

12. `abatch`: Resolves a Runnable for every input and processes the inputs asynchronously.

    When every configuration resolves to the default Runnable, execution is delegated directly to its optimized `abatch` implementation. Otherwise, invocations are gathered using the configured concurrency limit.

    * **Syntax:**
      ```python
      async abatch(
          self,
          inputs: list[Input], # Inputs to process
          config: RunnableConfig
          | list[RunnableConfig]
          | None = None, # Configuration for one or more inputs
          *,
          return_exceptions: bool = False, # Return exceptions instead of raising them
          **kwargs: Any | None # Additional invocation arguments
      ) -> list[Output]
      ```

13. `stream`: Resolves the configured Runnable and synchronously streams its output.
    * **Syntax:**
      ```python
      stream(
          self,
          input: Input, # Input passed to the selected Runnable
          config: RunnableConfig | None = None, # Runtime configuration
          **kwargs: Any | None # Additional streaming arguments
      ) -> Iterator[Output]
      ```

14. `astream`: Resolves the configured Runnable and asynchronously streams its output.
    * **Syntax:**
      ```python
      async astream(
          self,
          input: Input, # Input passed to the selected Runnable
          config: RunnableConfig | None = None, # Runtime configuration
          **kwargs: Any | None # Additional streaming arguments
      ) -> AsyncIterator[Output]
      ```

15. `transform`: Resolves the configured Runnable and synchronously transforms an iterator of inputs.
    * **Syntax:**
      ```python
      transform(
          self,
          input: Iterator[Input], # Iterator of inputs
          config: RunnableConfig | None = None, # Runtime configuration
          **kwargs: Any | None # Additional transformation arguments
      ) -> Iterator[Output]
      ```

16. `atransform`: Resolves the configured Runnable and asynchronously transforms an input stream.
    * **Syntax:**
      ```python
      async atransform(
          self,
          input: AsyncIterator[Input], # Asynchronous iterator of inputs
          config: RunnableConfig | None = None, # Runtime configuration
          **kwargs: Any | None # Additional transformation arguments
      ) -> AsyncIterator[Output]
      ```

17. `__getattr__`: Delegates missing attributes and callable methods to the default or configured Runnable.

    When a delegated method receives a configuration containing configurable values, the appropriate concrete Runnable is prepared before the method is called.

    * **Syntax:**
      ```python
      __getattr__(
          self,
          name: str # Name of the missing attribute
      ) -> Any
      ```

# RunnableConfigurableFields

`RunnableConfigurableFields` wraps a Runnable whose selected model fields can be changed at invocation time.

It is normally created through a Runnable's `configurable_fields` method.

## Bases

- `DynamicRunnable[Input, Output]`

## Attributes

1. `fields`: Maps field names on the default Runnable to their configurable-field specifications.
   * **Type:**
     ```python
     fields: dict[
         str,
         AnyConfigurableField
     ]
     ```

### Properties

1. `config_specs`: Returns the unique configuration specifications exposed by the configurable fields and the default Runnable.

   Plain configurable fields use the annotation, description, and default value of the corresponding Pydantic model field when these values are not supplied explicitly. Single-option and multi-option fields are converted into enum-based configuration specifications.

   * **Type:**
     ```python
     config_specs: list[ConfigurableFieldSpec]
     ```

### Methods

1. `configurable_fields`: Returns a new configurable-fields wrapper with the supplied specifications merged with the existing field specifications.
   * **Syntax:**
     ```python
     configurable_fields(
         self,
         **kwargs: AnyConfigurableField # Additional configurable field specifications
     ) -> RunnableSerializable[Input, Output]
     ```

2. `_prepare`: Applies configured field values to the default Runnable.

   Direct configurable values replace matching model fields. Single-option and multi-option values are resolved through their option mappings. When at least one field is configured, a new instance of the default Runnable's class is created.

   * **Syntax:**
     ```python
     _prepare(
         self,
         config: RunnableConfig | None = None # Configuration containing field selections
     ) -> tuple[
         Runnable[Input, Output],
         RunnableConfig
     ]
     ```

# StrEnum

`StrEnum` is a string-valued enumeration used to construct dynamic option types for configurable fields.

## Bases

- `str`
- `enum.Enum`

# RunnableConfigurableAlternatives

`RunnableConfigurableAlternatives` wraps a default Runnable and a set of alternative Runnables that can be selected at invocation time.

An alternative may be stored directly as a Runnable or provided through a zero-argument callable for lazy construction.

## Bases

- `DynamicRunnable[Input, Output]`

## Attributes

1. `which`: Stores the configurable field used to select the active Runnable.
   * **Type:**
     ```python
     which: ConfigurableField
     ```

2. `alternatives`: Maps alternative keys to Runnables or zero-argument factories.
   * **Type:**
     ```python
     alternatives: dict[
         str,
         Runnable[Input, Output]
         | Callable[
             [],
             Runnable[Input, Output]
         ]
     ]
     ```

3. `default_key`: Stores the option key representing the default Runnable.
   * **Type:**
     ```python
     default_key: str = "default"
     ```

4. `prefix_keys`: Controls whether nested configurable-field IDs are prefixed with the selected alternative namespace.

   A prefixed field uses the format `<which.id>==<alternative_key>/<field_id>`.

   * **Type:**
     ```python
     prefix_keys: bool
     ```

### Properties

1. `config_specs`: Returns the unique configuration specifications for selecting an alternative and configuring each available Runnable.

   The alternative selector is represented by a dynamically generated string enum. Nested configuration specifications are optionally prefixed by alternative key.

   * **Type:**
     ```python
     config_specs: list[ConfigurableFieldSpec]
     ```

### Methods

1. `configurable_fields`: Applies additional configurable fields to the default Runnable and returns a new alternatives wrapper.

   The existing selector, alternatives, default key, and prefix behavior are preserved.

   * **Syntax:**
     ```python
     configurable_fields(
         self,
         **kwargs: AnyConfigurableField # Configurable fields applied to the default Runnable
     ) -> RunnableSerializable[Input, Output]
     ```

2. `_prepare`: Selects the configured Runnable alternative and returns it with the prepared configuration.

   When `prefix_keys` is enabled, the namespace belonging to the selected alternative is removed from its configurable-field keys. The default Runnable is selected when no option is supplied. A `ValueError` is raised for an unknown alternative key.

   * **Syntax:**
     ```python
     _prepare(
         self,
         config: RunnableConfig | None = None # Configuration containing the alternative selection
     ) -> tuple[
         Runnable[Input, Output],
         RunnableConfig
     ]
     ```

## Functions

1. `prefix_config_spec`: Adds a namespace prefix to a configuration specification's identifier.

   Shared configuration specifications are returned unchanged.

   * **Syntax:**
     ```python
     prefix_config_spec(
         spec: ConfigurableFieldSpec, # Configuration specification to prefix
         prefix: str # Prefix added before the existing identifier
     ) -> ConfigurableFieldSpec
     ```

2. `make_options_spec`: Converts a single-option or multi-option configurable field into a `ConfigurableFieldSpec`.

   A dynamic string enum is created from the option keys. Single-option fields use that enum directly, while multi-option fields use a sequence of enum values.

   * **Syntax:**
     ```python
     make_options_spec(
         spec: ConfigurableFieldSingleOption
         | ConfigurableFieldMultiOption, # Option specification to convert
         description: str | None # Fallback description
     ) -> ConfigurableFieldSpec
     ```